In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

# Load MNIST data
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Preprocess data: resize to 64x64, normalize and expand dims for channels
x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

x_train = tf.image.resize(x_train, [64, 64]).numpy()
x_test = tf.image.resize(x_test, [64, 64]).numpy()

x_train = x_train / 255.0
x_test = x_test / 255.0

# Convert labels to categorical for multi-class classification (10 classes)
num_classes = 10
y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes)
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes)

# Define modified AlexNet for grayscale 64x64 images
def create_alexnet():
    model = models.Sequential([
        # First Convolutional Block - reduced kernel size and stride
        layers.Conv2D(96, kernel_size=(7, 7), strides=2, activation='relu', input_shape=(64, 64, 1)),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=(2, 2), strides=2),

        # Second Convolutional Block
        layers.Conv2D(256, kernel_size=(5, 5), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=(2, 2), strides=2),

        # Third Convolutional Block
        layers.Conv2D(384, kernel_size=(3, 3), padding='same', activation='relu'),
        layers.Conv2D(384, kernel_size=(3, 3), padding='same', activation='relu'),
        layers.Conv2D(256, kernel_size=(3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D(pool_size=(2, 2), strides=2),

        # Fully Connected Layers
        layers.Flatten(),
        layers.Dense(2048, activation='relu'),  # Reduced from 4096
        layers.Dropout(0.5),
        layers.Dense(1024, activation='relu'),  # Reduced from 4096
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

model = create_alexnet()
model.summary()

# Train model
model.fit(x_train, y_train_cat, epochs=10, batch_size=128, validation_split=0.1, verbose=2)

# Evaluate model on test data
test_loss, test_acc = model.evaluate(x_test, y_test_cat, verbose=2)
print(f"\nTest accuracy: {test_acc * 100:.2f}%")

# Make predictions
y_pred_prob = model.predict(x_test)
y_pred = np.argmax(y_pred_prob, axis=1)

# Performance evaluation
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, digits=4))

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_5 (Conv2D)               │ (None, 29, 29, 96)     │         4,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 29, 29, 96)     │           384 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 96)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 14, 14, 256)    │       614,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 14, 14, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 7, 7, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 7, 7, 384)      │       885,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 7, 7, 384)      │     1,327,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 7, 7, 256)      │       884,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 3, 3, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 2304)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2048)           │     4,720,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1024)           │     2,098,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │        10,250 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,547,530 (40.24 MB)

 Trainable params: 10,546,826 (40.23 MB)

 Non-trainable params: 704 (2.75 KB)

Epoch 1/10
422/422 - 319s - 755ms/step - accuracy: 0.9312 - loss: 0.2206 - val_accuracy: 0.9218 - val_loss: 0.2520
Epoch 2/10
422/422 - 319s - 755ms/step - accuracy: 0.9312 - loss: 0.2206 - val_accuracy: 0.9218 - val_loss: 0.2520
Epoch 2/10
422/422 - 323s - 764ms/step - accuracy: 0.9833 - loss: 0.0592 - val_accuracy: 0.9862 - val_loss: 0.0548
Epoch 3/10
422/422 - 323s - 764ms/step - accuracy: 0.9833 - loss: 0.0592 - val_accuracy: 0.9862 - val_loss: 0.0548
Epoch 3/10


In [5]:
model.fit(x_train, y_train_cat, epochs=5, batch_size=128, validation_split=0.1)

Epoch 1/5


ValueError: Exception encountered when calling MaxPooling2D.call().

[1mNegative dimension size caused by subtracting 3 from 2 for '{{node sequential_1/max_pooling2d_2_1/MaxPool2d}} = MaxPool[T=DT_FLOAT, data_format="NHWC", explicit_paddings=[], ksize=[1, 3, 3, 1], padding="VALID", strides=[1, 2, 2, 1]](sequential_1/conv2d_4_1/Relu)' with input shapes: [?,2,2,256].[0m

Arguments received by MaxPooling2D.call():
  • inputs=tf.Tensor(shape=(None, 2, 2, 256), dtype=float32)